In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# load data
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
df.drop('customerID', axis=1, inplace=True)

# new features
df['AvgMonthlySpend'] = df['TotalCharges'] / (df['tenure'] + 1)
service_cols = ['PhoneService', 'OnlineSecurity', 'OnlineBackup',
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['ServiceCount'] = df[service_cols].apply(lambda x: (x == 'Yes').sum(), axis=1)
df['IsMonthly'] = (df['Contract'] == 'Month-to-month').astype(int)

# encode
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService',
               'PaperlessBilling', 'Churn']
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

multi_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity',
              'OnlineBackup', 'DeviceProtection', 'TechSupport',
              'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']
df = pd.get_dummies(df, columns=multi_cols)

# target is MonthlyCharges for regression
x = df.drop('MonthlyCharges', axis=1)
y = df['MonthlyCharges']

# scale
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)
x_scaled = pd.DataFrame(x_scaled, columns=x.columns)

# split 70/15/15
x_train, x_temp, y_train, y_temp = train_test_split(
    x_scaled, y, test_size=0.30, random_state=42)
x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, test_size=0.50, random_state=42)

print("Data ready!")
print("Train:", x_train.shape, "Val:", x_val.shape, "Test:", x_test.shape)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# function to print results
def show_results(name, y_test, y_pred):
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    n = len(y_test)
    p = x_test.shape[1]
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    print(f"\n{name}")
    print(f"MAE: {mae:.3f}  MSE: {mse:.3f}  RMSE: {rmse:.3f}  R2: {r2:.3f}  Adj R2: {adj_r2:.3f}")
    return [name, round(mae,3), round(mse,3), round(rmse,3), round(r2,3), round(adj_r2,3)]

results = []

# linear regression
lr = LinearRegression()
lr.fit(x_train, y_train)
y_pred_lr = lr.predict(x_test)
results.append(show_results('Linear Regression', y_test, y_pred_lr))

# ridge
ridge = Ridge(alpha=1.0)
ridge.fit(x_train, y_train)
y_pred_ridge = ridge.predict(x_test)
results.append(show_results('Ridge', y_test, y_pred_ridge))

# lasso
lasso = Lasso(alpha=0.1)
lasso.fit(x_train, y_train)
y_pred_lasso = lasso.predict(x_test)
results.append(show_results('Lasso', y_test, y_pred_lasso))

# elasticnet
en = ElasticNet(alpha=0.1, l1_ratio=0.5)
en.fit(x_train, y_train)
y_pred_en = en.predict(x_test)
results.append(show_results('ElasticNet', y_test, y_pred_en))

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

# decision tree
params = {'max_depth': [3, 5, 7, 10]}
dt = DecisionTreeRegressor(random_state=42)
grid_dt = GridSearchCV(dt, params, cv=3, scoring='r2')
grid_dt.fit(x_train, y_train)
best_dt = grid_dt.best_estimator_
y_pred_dt = best_dt.predict(x_test)
results.append(show_results('Decision Tree', y_test, y_pred_dt))

# random forest
params = {'n_estimators': [50, 100], 'max_depth': [5, 10]}
rf = RandomForestRegressor(random_state=42)
grid_rf = GridSearchCV(rf, params, cv=3, scoring='r2')
grid_rf.fit(x_train, y_train)
best_rf = grid_rf.best_estimator_
y_pred_rf = best_rf.predict(x_test)
results.append(show_results('Random Forest', y_test, y_pred_rf))

In [ ]:
from sklearn.svm import SVR

# smaller grid to run faster
params = {'C': [1, 10], 'kernel': ['linear']}
svr = SVR()
grid_svr = GridSearchCV(svr, params, cv=3, scoring='r2')
grid_svr.fit(x_train, y_train)
best_svr = grid_svr.best_estimator_
y_pred_svr = best_svr.predict(x_test)
results.append(show_results('SVR', y_test, y_pred_svr))

In [ ]:
# show all results as table
table = pd.DataFrame(results, columns=['Model', 'MAE', 'MSE', 'RMSE', 'R2', 'Adj R2'])
print(table.to_string(index=False))

In [ ]:
# plot actual vs predicted for each model
models = [('Linear Regression', y_pred_lr),
          ('Ridge', y_pred_ridge),
          ('Lasso', y_pred_lasso),
          ('ElasticNet', y_pred_en),
          ('Decision Tree', y_pred_dt),
          ('Random Forest', y_pred_rf),
          ('SVR', y_pred_svr)]

for name, y_pred in models:
    plt.figure(figsize=(5,4))
    plt.scatter(y_test, y_pred, alpha=0.4, color='steelblue')
    plt.plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()], 'r--')
    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.title(f'{name} - Actual vs Predicted')
    plt.show()